# H1 Operasyonel Aciliyet Skoru — V5-Bağımsız Yeniden İnşa

**Amaç:** `panel_data/smart_maintenance.json`'ı ML V5 modeline hiç başvurmadan, sadece ham arıza verisi (`ariza_model.csv`) ve `ciddiyet_skoru` ile yeniden üretmek.

**Neden:** V5 model artıklarından kurtulup panel'in H1 retroaktif aciliyet görüntülerini saf operasyonel mantıkla beslemek.

**Kaynak:** `panel_data/temiz_veri/ariza_model.csv` (58,559 arıza × 3,509 araç, H1 2025)

**Aciliyet skoru formülü (Seçenek 1: saf ciddiyet_skoru-tabanlı):**
```
aciliyet_skoru_raw = avg_ciddiyet_90 × 10
                   + agir_30          × 5
                   + recent_intensity × 20
    recent_intensity = cnt_30 / cnt_total_h1  (cnt_total_h1=0 ise 0)
aciliyet_skoru = min-max normalize → [0, 100]
```

**Tier eşikleri:** Quartile-tabanlı (data-driven, V5 dağılımına bakmadan)
- KRİTİK: skor ≥ p85
- YÜKSEK: p65 ≤ skor < p85
- ORTA: p35 ≤ skor < p65
- DÜŞÜK: skor < p35

In [ ]:
import pandas as pd
import numpy as np
import json
from collections import Counter
from pathlib import Path

DATATHON_DIR = Path(r'c:\Users\asus\Desktop\Datathon')
PANEL_DIR = DATATHON_DIR / 'panel_data'

# --- KAYNAK: V5 ham veriye dokunmuyor, bu dosya CIDDIYET_SKORU.ipynb tarafından üretiliyor ---
df = pd.read_csv(PANEL_DIR / 'temiz_veri' / 'ariza_model.csv')
df['OLAYTARIHI'] = pd.to_datetime(df['OLAYTARIHI'])
print(f'Yuklenen arıza kaydı: {len(df):,}')
print(f'Araç sayısı: {df["KAPINO"].nunique():,}')
print(f'Tarih aralığı: {df["OLAYTARIHI"].min()} → {df["OLAYTARIHI"].max()}')
print(f'ciddi_ariza dağılımı:\n{df["ciddi_ariza"].value_counts()}')

In [ ]:
# H1 referans tarihi = veri sonu
REF_DATE = df['OLAYTARIHI'].max()
WIN30 = REF_DATE - pd.Timedelta(days=30)
WIN90 = REF_DATE - pd.Timedelta(days=90)
print(f'Referans: {REF_DATE} | 30g: ≥{WIN30.date()} | 90g: ≥{WIN90.date()}')

df['_son30'] = df['OLAYTARIHI'] >= WIN30
df['_son90'] = df['OLAYTARIHI'] >= WIN90

In [ ]:
# --- ARAÇ-TİPİ MAPPING (HATCINSI majority) ---
def _map_arac_tipi(s):
    s = str(s).upper()
    if 'METROB' in s: return 'Metrobüs'
    if 'ELEKTR' in s: return 'Elektrikli'
    return 'Otobüs'

# Araç başına en çok geçen HATCINSI
hat_majority = (df.groupby('KAPINO')['HATCINSI']
                  .agg(lambda s: s.value_counts().idxmax())
                  .map(_map_arac_tipi))
print('arac_tipi dağılımı:'); print(hat_majority.value_counts())

In [ ]:
# --- ARAÇ BAŞINA AGGREGASYON ---
VERI_YIL = 2025  # H1 referans yılı

def _agg_arac(g):
    son90 = g[g['_son90']]
    son30 = g[g['_son30']]
    ciddi_g = g[g['ciddi_ariza'] == 1]

    avg_ciddiyet_90 = son90['ciddiyet_skoru'].mean() if len(son90) else 0.0
    agir_30 = int((son30['ciddi_ariza'] == 1).sum())
    cnt_30 = int(len(son30))
    cnt_total_h1 = int(len(g))
    ciddi_total_h1 = int((g['ciddi_ariza'] == 1).sum())
    recent_intensity = (cnt_30 / cnt_total_h1) if cnt_total_h1 > 0 else 0.0

    # ciddi/total operasyonel olasılık (V5 ML değil, basit oran)
    ciddi_olasilik_pct = round((ciddi_total_h1 / cnt_total_h1 * 100) if cnt_total_h1 > 0 else 0, 1)
    # Çoklu ciddi: birden fazla ciddi arıza yapma yatkınlığı (operasyonel proxy)
    coklu_ciddi_olasilik = round((1 - 1 / (1 + ciddi_total_h1 / 3)) * 100, 1) if ciddi_total_h1 > 0 else 0.0

    # Son ciddi olay
    if len(ciddi_g):
        son_c = ciddi_g.sort_values('OLAYTARIHI').iloc[-1]
        son_ciddi_tarih = son_c['OLAYTARIHI'].strftime('%Y-%m-%d %H:%M:%S')
        son_ciddi_neden = str(son_c['ARIZAUSTKODTANIM'])
    else:
        son_ciddi_tarih = None
        son_ciddi_neden = None

    # Tekrarlayan: H1'de en çok yapılan ARIZAUSTKODTANIM
    if len(g):
        tekrarlayan_neden = str(g['ARIZAUSTKODTANIM'].value_counts().idxmax())
    else:
        tekrarlayan_neden = None

    # Son 90 günde farklı ciddi kategori sayısı
    ciddi_kat_cesit_90 = int(son90[son90['ciddi_ariza'] == 1]['ARIZAUSTKODTANIM'].nunique())

    # Kaskad ortalaması: ardışık arızalar arası gün farkı ortalaması
    tarihler = g['OLAYTARIHI'].sort_values()
    if len(tarihler) >= 2:
        kaskad_ort = float(tarihler.diff().dt.total_seconds().dropna().mean() / 86400)
        kaskad_ort = round(kaskad_ort, 2)
    else:
        kaskad_ort = None

    # Tepki/müdahale: TOPLAM_SURE_DK ortalaması (proxy)
    tepki_dk = round(float(g['TOPLAM_SURE_DK'].mean()), 1) if 'TOPLAM_SURE_DK' in g.columns and len(g) else 0.0

    # Statik araç bilgisi: ilk kayıttan
    first = g.iloc[0]
    arac_yasi = int(VERI_YIL - first['MODELYILI']) if pd.notna(first['MODELYILI']) else None

    return pd.Series({
        'marka':            str(first['MARKA']) if pd.notna(first['MARKA']) else None,
        'model':            str(first['MODEL']) if pd.notna(first['MODEL']) else None,
        'arac_cinsi':       str(first['ARACCINSI']) if pd.notna(first['ARACCINSI']) else None,
        'arac_yasi':        arac_yasi,
        'garaj':            str(first['GARAJ']) if pd.notna(first['GARAJ']) else None,
        'yakit_turu':       str(first['YAKITTURU']) if pd.notna(first['YAKITTURU']) else None,
        'avg_ciddiyet_90':  round(float(avg_ciddiyet_90), 3),
        'agir_30':          agir_30,
        'cnt_30':           cnt_30,
        'cnt_total_h1':     cnt_total_h1,
        'ciddi_total_h1':   ciddi_total_h1,
        'recent_intensity': round(float(recent_intensity), 4),
        'ciddi_olasilik_pct':    ciddi_olasilik_pct,
        'coklu_ciddi_olasilik':  coklu_ciddi_olasilik,
        'son_ciddi_tarih':       son_ciddi_tarih,
        'son_ciddi_neden':       son_ciddi_neden,
        'tekrarlayan_neden':     tekrarlayan_neden,
        'ciddi_kat_cesit_90':    ciddi_kat_cesit_90,
        'kaskad_ort':            kaskad_ort,
        'tepki_dk':              tepki_dk,
    })

agg = df.groupby('KAPINO', sort=False).apply(_agg_arac).reset_index()
agg['arac_tipi'] = agg['KAPINO'].map(hat_majority)
print(f'Agregasyon tamamlandı: {len(agg):,} araç')
agg.head(3)

In [ ]:
# --- ACİLİYET SKORU FORMÜLÜ (Seçenek 1: saf ciddiyet_skoru-tabanlı) ---
raw = (agg['avg_ciddiyet_90'] * 10
       + agg['agir_30']          * 5
       + agg['recent_intensity'] * 20)

# 0-100 min-max normalize
r_min, r_max = raw.min(), raw.max()
agg['aciliyet_skoru'] = ((raw - r_min) / (r_max - r_min) * 100).round(2)

print('Aciliyet skoru dağılımı:')
print(agg['aciliyet_skoru'].describe())

# Tier eşikleri: quantile-based
p35, p65, p85 = agg['aciliyet_skoru'].quantile([0.35, 0.65, 0.85]).values
print(f'\nTier eşikleri: p35={p35:.2f} | p65={p65:.2f} | p85={p85:.2f}')

def _tier(s):
    if s >= p85: return 'KRİTİK'
    if s >= p65: return 'YÜKSEK'
    if s >= p35: return 'ORTA'
    return 'DÜŞÜK'

agg['aciliyet_tier'] = agg['aciliyet_skoru'].map(_tier)
print('\nTier dağılımı:')
print(agg['aciliyet_tier'].value_counts())

In [ ]:
# --- BAKIM UYARISI METNİ (kural-tabanlı) ---
def _bakim_uyari(row):
    parts = []
    t = row['aciliyet_tier']
    if t == 'KRİTİK':
        parts.append('Acil bakima alinmasi onerilir')
    elif t == 'YÜKSEK':
        parts.append('Yakin takip onerilir')
    elif t == 'ORTA':
        parts.append('Rutin bakim takvimine alinabilir')
    else:
        parts.append('Mevcut bakim takvimi yeterli')

    if row['tekrarlayan_neden']:
        parts.append(f"TEKRARLAYAN: {row['tekrarlayan_neden']}")
    if row['ciddi_kat_cesit_90'] >= 3:
        parts.append(f"Son 90 gunde {row['ciddi_kat_cesit_90']} farkli kategori")
    return ' | '.join(parts)

agg['bakim_uyarisi'] = agg.apply(_bakim_uyari, axis=1)
agg['bakim_uyarisi'].head(3).tolist()

In [ ]:
# --- JSON ÇIKTI ŞEMASI (panel'in beklediği 24 kolon) ---
def _row_to_dict(r):
    # Tahmin güvenilir = en az 3 arıza kaydı varsa istatistiksel olarak makul
    guvenilir = 'EVET' if r['cnt_total_h1'] >= 3 else 'HAYIR'
    return {
        'kapino':                str(r['KAPINO']),
        'arac_tipi':             r['arac_tipi'] or 'Otobüs',
        'marka':                 r['marka'],
        'model':                 r['model'],
        'arac_cinsi':            r['arac_cinsi'],
        'arac_yasi':             int(r['arac_yasi']) if pd.notna(r['arac_yasi']) else None,
        'garaj':                 r['garaj'],
        'yakit_turu':            r['yakit_turu'],
        'aciliyet_tier':         r['aciliyet_tier'],
        'aciliyet_skoru':        float(r['aciliyet_skoru']),
        'ciddi_olasilik_pct':    float(r['ciddi_olasilik_pct']),
        'coklu_ciddi_olasilik':  float(r['coklu_ciddi_olasilik']),
        'son_ciddi_tarih':       r['son_ciddi_tarih'],
        'son_ciddi_neden':       r['son_ciddi_neden'],
        'tekrarlayan_neden':     r['tekrarlayan_neden'],
        'ciddi_kat_cesit_90':    int(r['ciddi_kat_cesit_90']),
        'agir_30':               int(r['agir_30']),
        'cnt_30':                int(r['cnt_30']),
        'cnt_total_h1':          int(r['cnt_total_h1']),
        'ciddi_total_h1':        int(r['ciddi_total_h1']),
        'mevsim':                '2025-H1',
        'bakim_uyarisi':         r['bakim_uyarisi'],
        'tahmin_guvenilir':      guvenilir,
        'kaynak':                'H1 ham veri agregasyonu (ciddiyet_skoru tabanlı)',
    }

records = [_row_to_dict(r) for _, r in agg.iterrows()]
print(f'JSON kayıt: {len(records):,}')
print('Ornek kayit:')
print(json.dumps(records[0], ensure_ascii=False, indent=2))

In [ ]:
# --- YAZIM: V5 backup + yeni JSON ---
OUT_PATH = PANEL_DIR / 'smart_maintenance.json'
BACKUP_DIR = PANEL_DIR / '_arsiv_v5'
BACKUP_DIR.mkdir(exist_ok=True)
BACKUP_PATH = BACKUP_DIR / 'smart_maintenance_v5.json'

# 1) Eski V5 çıktısını yedekle
if OUT_PATH.exists() and not BACKUP_PATH.exists():
    import shutil
    shutil.copy2(OUT_PATH, BACKUP_PATH)
    print(f'V5 backup: {BACKUP_PATH}')
elif BACKUP_PATH.exists():
    print(f'V5 backup zaten mevcut: {BACKUP_PATH}')

# 2) Yeni JSON'u yaz
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f'Yeni smart_maintenance.json yazıldı: {OUT_PATH} ({OUT_PATH.stat().st_size/1024:.0f} KB)')

# 3) iett_panel'in panel_data klasörüne de kopyala
import shutil
PANEL_LIVE = Path(r'c:\Users\asus\Desktop\iett_panel\panel_data\smart_maintenance.json')
if PANEL_LIVE.parent.exists():
    PANEL_LIVE_BACKUP = PANEL_LIVE.parent / '_arsiv_v5'
    PANEL_LIVE_BACKUP.mkdir(exist_ok=True)
    if PANEL_LIVE.exists() and not (PANEL_LIVE_BACKUP / 'smart_maintenance_v5.json').exists():
        shutil.copy2(PANEL_LIVE, PANEL_LIVE_BACKUP / 'smart_maintenance_v5.json')
    shutil.copy2(OUT_PATH, PANEL_LIVE)
    print(f'Panel canli kopyasi guncellendi: {PANEL_LIVE}')

In [ ]:
# --- DOĞRULAMA: yeni vs eski tier dağılımı + örnek araç karşılaştırma ---
with open(BACKUP_PATH, encoding='utf-8') as f:
    eski = json.load(f)

with open(OUT_PATH, encoding='utf-8') as f:
    yeni = json.load(f)

print(f'Kayıt sayısı: ESKI={len(eski)} | YENI={len(yeni)}')
print(f'\nTier dağılımı (ESKI V5):'); print(Counter(r['aciliyet_tier'] for r in eski))
print(f'\nTier dağılımı (YENI H1):'); print(Counter(r['aciliyet_tier'] for r in yeni))

# Aynı araç için skor farkı
eski_idx = {r['kapino']: r for r in eski}
diffs = []
for r in yeni:
    e = eski_idx.get(r['kapino'])
    if e:
        diffs.append({
            'kapino': r['kapino'],
            'eski_tier': e['aciliyet_tier'], 'yeni_tier': r['aciliyet_tier'],
            'eski_skor': e['aciliyet_skoru'], 'yeni_skor': r['aciliyet_skoru'],
        })

diff_df = pd.DataFrame(diffs)
tier_match = (diff_df['eski_tier'] == diff_df['yeni_tier']).mean() * 100
print(f'\nTier eşleşme oranı (yeni vs eski): %{tier_match:.1f}')
print(f'Skor Pearson korelasyon: {diff_df[["eski_skor","yeni_skor"]].corr().iloc[0,1]:.3f}')

## Çıktı

- `panel_data/smart_maintenance.json` → V5-bağımsız yeniden üretildi (3,509 araç)
- `panel_data/_arsiv_v5/smart_maintenance_v5.json` → eski V5 çıktısı (geri dönüş için)
- `iett_panel/panel_data/smart_maintenance.json` → canlı panel'e kopyalandı

**Şema:** Panel'in beklediği 24 kolon birebir korundu — UI değişikliği gerekmez.

**Formül:** Saf ciddiyet_skoru-tabanlı. ML modeline başvurmaz. Her araç için H1 (Oca-Haz 2025) ham arıza verisinden hesaplanır.